# Modul 2: Backpropagation dan Automatic Differentiation

**Nama: Adil Aulia Rahma Nurhidayah**  
**NIM: 122450058** 
**Kelas: RA**  
**Tanggal: 23-9-2026**  

Simpan berkas ini sebagai `M02_NIM.ipynb` sebelum mulai mengerjakan.

## Petunjuk

1. Ganti seluruh penanda `TODO`. Jangan menghapus sel pemeriksaan.
2. Nilai Kasus 1 tidak boleh diubah; seluruh angka pada modul mengacu padanya.
3. Gunakan `float64` untuk semua perhitungan gradien.
4. Tuliskan turunan manual pada sel markdown, bukan hanya di kertas.
5. Notebook harus lolos *Restart Kernel and Run All* sebelum dikumpulkan.
6. Luaran: `M02_NIM.ipynb`, `M02_NIM.pdf`, dan `M02_NIM_metrics.csv`.

In [1]:
import platform
import random

import numpy as np
import pandas as pd
import torch
from torch import nn

NIM = '122450058'
SEED = int(str(NIM)[-4:]) if str(NIM).isdigit() else 42   # seed individual
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(SEED)
torch.set_default_dtype(torch.float64)
pd.set_option('display.precision', 8)
print({'python': platform.python_version(), 'numpy': np.__version__,
       'torch': torch.__version__, 'device': str(DEVICE), 'seed': SEED})

{'python': '3.11.9', 'numpy': '2.4.6', 'torch': '2.14.0+cpu', 'device': 'cpu', 'seed': 58}


# A. Pre-lab - 10 poin

Jawab sebelum sesi praktikum dimulai.

## 1. Gradien lokal vs gradien total pada satu simpul

**Gradien lokal** adalah turunan parsial keluaran suatu simpul terhadap **masukan langsungnya saja**, dengan variabel lain dianggap konstan. Ia hanya melihat operasi pada simpul itu sendiri, bukan seluruh jaringan.

**Gradien total** adalah turunan *loss* akhir $\mathcal{L}$ terhadap suatu variabel, diperoleh dengan mengalikan gradien lokal sepanjang jalur *chain rule* dari $\mathcal{L}$ mundur hingga variabel tersebut (dijumlahkan bila jalurnya bercabang).

Contoh pada simpul $z = w\cdot x + b$:

$$
\frac{\partial z}{\partial w} = x, \qquad
\frac{\partial z}{\partial x} = w, \qquad
\frac{\partial z}{\partial b} = 1
$$

Gradien total $\partial\mathcal{L}/\partial w$ baru diperoleh setelah dikalikan dengan $\partial\mathcal{L}/\partial z$ dari simpul berikutnya.

---

## 2. Mengapa `backward()` hanya dapat dipanggil pada tensor skalar

Gradien secara matematis didefinisikan sebagai laju perubahan sebuah **nilai skalar** terhadap variabel-variabelnya. Jika keluaran berupa vektor, "gradien" tidak terdefinisi tunggal — dibutuhkan vektor bobot untuk mengalikannya (*vector-Jacobian product*).

PyTorch karena itu mensyaratkan tensor tujuan `backward()` bersifat skalar, atau secara eksplisit diberi argumen `gradient=` berbentuk sama. Pada praktik *deep learning*, *loss* selalu diskalarkan menjadi satu angka (rata-rata atau jumlah), sehingga syarat ini terpenuhi secara alami.

---

## 3. Isi `.grad` bila `backward()` dipanggil dua kali tanpa `zero_grad()`

PyTorch **mengakumulasi** (menjumlahkan) gradien, bukan menimpanya. Bila `backward()` dipanggil dua kali tanpa `zero_grad()` di antaranya:

$$
\texttt{.grad} = \nabla_1 + \nabla_2
$$

yaitu gradien pemanggilan pertama **ditambah** gradien pemanggilan kedua. Jika kedua pemanggilan berasal dari *loss* dan *graph* yang sama, hasilnya menjadi $2\nabla$.

Perilaku akumulasi ini disengaja untuk mendukung *gradient accumulation* pada batch besar, tetapi menjadi *bug* bila lupa memanggil `zero_grad()` setiap iterasi.

---

## 4. Mengapa turunan BCE-with-logits terhadap logit berbentuk $p - y$, bukan $-y/p$

BCE biasa didefinisikan pada **probabilitas** $p = \sigma(z)$:

$$
\text{BCE}(y, p) = -\big[\,y\log p + (1-y)\log(1-p)\,\big]
$$

Turunannya terhadap $p$ memang:

$$
\frac{\partial \text{BCE}}{\partial p} = -\frac{y}{p} + \frac{1-y}{1-p}
$$

Namun `BCEWithLogitsLoss` bekerja pada **logit** $z$ (sebelum sigmoid). Dengan aturan rantai dan $\dfrac{\partial p}{\partial z} = p(1-p)$:

$$
\frac{\partial \text{BCE}}{\partial z}
= \frac{\partial \text{BCE}}{\partial p}\cdot\frac{\partial p}{\partial z}
= \left(-\frac{y}{p} + \frac{1-y}{1-p}\right) p(1-p)
= -y(1-p) + (1-y)p
= p - y
$$

Jadi bentuk $p - y$ adalah hasil **penyederhanaan** setelah dikalikan turunan sigmoid. Bentuk ini jauh lebih stabil secara numerik karena tidak membagi dengan $p$ yang bisa sangat kecil (menghindari *overflow/underflow*) — itulah alasan PyTorch menyediakan versi *logits*.

---

## Graf komputasi Kasus 1

Urutan simpul dari $\mathbf{x}$ sampai $\mathcal{L}$ (linear → ReLU → linear → BCE-logits), beserta gradien lokal tiap simpul:

$$
\mathbf{x}
\;\xrightarrow{\;\text{lin}_1\;}\;
\mathbf{z}_1 = W_1\mathbf{x} + \mathbf{b}_1
\;\xrightarrow{\;\text{ReLU}\;}\;
\mathbf{a}_1 = \max(0,\mathbf{z}_1)
\;\xrightarrow{\;\text{lin}_2\;}\;
\mathbf{z}_2 = W_2\mathbf{a}_1 + \mathbf{b}_2
\;\xrightarrow{\;\text{BCE-logits}\;}\;
\mathcal{L}
$$

### Gradien lokal di tiap simpul

| Simpul | Operasi | Gradien lokal |
|---|---|---|
| $\text{lin}_1$ | $\mathbf{z}_1 = W_1\mathbf{x} + \mathbf{b}_1$ | $\dfrac{\partial \mathbf{z}_1}{\partial W_1} = \mathbf{x}^\top,\quad \dfrac{\partial \mathbf{z}_1}{\partial \mathbf{b}_1} = \mathbf{1},\quad \dfrac{\partial \mathbf{z}_1}{\partial \mathbf{x}} = W_1^\top$ |
| ReLU | $\mathbf{a}_1 = \max(0,\mathbf{z}_1)$ | $\dfrac{\partial \mathbf{a}_1}{\partial \mathbf{z}_1} = \mathbb{1}[\mathbf{z}_1 > 0]$ |
| $\text{lin}_2$ | $\mathbf{z}_2 = W_2\mathbf{a}_1 + \mathbf{b}_2$ | $\dfrac{\partial \mathbf{z}_2}{\partial W_2} = \mathbf{a}_1^\top,\quad \dfrac{\partial \mathbf{z}_2}{\partial \mathbf{b}_2} = \mathbf{1},\quad \dfrac{\partial \mathbf{z}_2}{\partial \mathbf{a}_1} = W_2^\top$ |
| BCE-logits | $\mathcal{L} = \text{BCEWithLogits}(y,\mathbf{z}_2)$ | $\dfrac{\partial \mathcal{L}}{\partial \mathbf{z}_2} = \sigma(\mathbf{z}_2) - y = \mathbf{p} - y$ |

### Gradien total (contoh untuk $W_1$)

$$
\frac{\partial \mathcal{L}}{\partial W_1}
= \underbrace{(\mathbf{p}-y)}_{\partial\mathcal{L}/\partial\mathbf{z}_2}
\cdot \underbrace{W_2^\top}_{\partial\mathbf{z}_2/\partial\mathbf{a}_1}
\cdot \underbrace{\mathbb{1}[\mathbf{z}_1>0]}_{\partial\mathbf{a}_1/\partial\mathbf{z}_1}
\cdot \underbrace{\mathbf{x}^\top}_{\partial\mathbf{z}_1/\partial W_1}
$$

# B. Turunan manual - 20 poin

## Setup Kasus 1

$$
\mathbf{x}=\begin{bmatrix}2 & -1\end{bmatrix},\quad
\mathbf{W}^{(1)}=\begin{bmatrix}0.5 & -0.5\\ 1 & 1\end{bmatrix},\quad
\mathbf{b}^{(1)}=\begin{bmatrix}0 & 0\end{bmatrix},\quad
\mathbf{W}^{(2)}=\begin{bmatrix}2 & -1\end{bmatrix},\quad
b^{(2)}=0.5,\quad y=1
$$

**Forward pass:**

$$
\mathbf{z}^{(1)} = \mathbf{x}\,\mathbf{W}^{(1)\top} + \mathbf{b}^{(1)}
= \begin{bmatrix}1.5 & 1\end{bmatrix}
$$

$$
\mathbf{h} = \max(0,\mathbf{z}^{(1)}) = \begin{bmatrix}1.5 & 1\end{bmatrix}
$$

$$
z^{(2)} = \mathbf{h}\,\mathbf{W}^{(2)\top} + b^{(2)} = 2.5
$$

$$
p = \sigma(2.5) \approx 0.92414182, \qquad
\mathcal{L} = -\log p \approx 0.07888973
$$

---

## Penurunan berurutan

**1.** $\partial\mathcal{L}/\partial z^{(2)} = p - y = 0.92414182 - 1 = -0.07585818$  (skalar)

**2.** $\partial\mathcal{L}/\partial \mathbf{W}^{(2)} = \dfrac{\partial\mathcal{L}}{\partial z^{(2)}}\,\mathbf{h} = -0.07585818 \begin{bmatrix}1.5 & 1\end{bmatrix} = \begin{bmatrix}-0.11378727 & -0.07585818\end{bmatrix}$  (shape `(1,2)`)

**3.** $\partial\mathcal{L}/\partial b^{(2)} = \dfrac{\partial\mathcal{L}}{\partial z^{(2)}} = -0.07585818$  (skalar)

**4.** $\partial\mathcal{L}/\partial \mathbf{h} = \dfrac{\partial\mathcal{L}}{\partial z^{(2)}}\,\mathbf{W}^{(2)} = -0.07585818\begin{bmatrix}2 & -1\end{bmatrix} = \begin{bmatrix}-0.15171636 & 0.07585818\end{bmatrix}$  (shape `(1,2)`)

**5.** $\partial\mathcal{L}/\partial \mathbf{z}^{(1)} = \dfrac{\partial\mathcal{L}}{\partial \mathbf{h}} \odot \mathbb{1}[\mathbf{z}^{(1)}>0] = \begin{bmatrix}-0.15171636 & 0.07585818\end{bmatrix}$  (shape `(1,2)`)

**6.** $\partial\mathcal{L}/\partial \mathbf{W}^{(1)} = \left(\dfrac{\partial\mathcal{L}}{\partial \mathbf{z}^{(1)}}\right)^{\!\top}\mathbf{x} = \begin{bmatrix}-0.30343272 & 0.15171636\\ 0.15171636 & -0.07585818\end{bmatrix}$  (shape `(2,2)`)

**7.** $\partial\mathcal{L}/\partial \mathbf{b}^{(1)} = \dfrac{\partial\mathcal{L}}{\partial \mathbf{z}^{(1)}} = \begin{bmatrix}-0.15171636 & 0.07585818\end{bmatrix}$  (shape `(1,2)`)

---

## Ringkasan shape

| No | Gradien | Shape |
|----|---------|-------|
| 1 | $\partial\mathcal{L}/\partial z^{(2)}$ | skalar |
| 2 | $\partial\mathcal{L}/\partial \mathbf{W}^{(2)}$ | `(1,2)` |
| 3 | $\partial\mathcal{L}/\partial b^{(2)}$ | skalar |
| 4 | $\partial\mathcal{L}/\partial \mathbf{h}$ | `(1,2)` |
| 5 | $\partial\mathcal{L}/\partial \mathbf{z}^{(1)}$ | `(1,2)` |
| 6 | $\partial\mathcal{L}/\partial \mathbf{W}^{(1)}$ | `(2,2)` |
| 7 | $\partial\mathcal{L}/\partial \mathbf{b}^{(1)}$ | `(1,2)` |

In [5]:
x  = np.array([2.0, -1.0])
W1 = np.array([[0.5, -0.5], [1.0, 1.0]])
b1 = np.array([0.0, 0.0])
W2 = np.array([2.0, -1.0])
b2 = 0.5
y  = 1.0

def forward(x, W1, b1, W2, b2, y):
    """TODO 1: kembalikan dict berisi z1, h, z2, p, dan loss."""
    # Linear 1: z1 = x @ W1.T + b1   -> shape (2,)
    z1 = x @ W1.T + b1

    # ReLU: h = max(0, z1)           -> shape (2,)
    h = np.maximum(0.0, z1)

    # Linear 2: z2 = h @ W2.T + b2   -> skalar
    z2 = h @ W2.T + b2

    # Sigmoid: p = sigma(z2)         -> skalar
    p = 1.0 / (1.0 + np.exp(-z2))

    # BCE loss: L = -[y log p + (1-y) log(1-p)]
    loss = -(y * np.log(p) + (1.0 - y) * np.log(1.0 - p))

    return {'z1': z1, 'h': h, 'z2': z2, 'p': p, 'loss': loss}

nilai = forward(x, W1, b1, W2, b2, y)
print({k: np.round(v, 6) for k, v in nilai.items()})

# Pemeriksaan wajib: jangan diubah.
assert np.allclose(nilai['z1'], [1.5, 1.0]), 'z1 belum benar'
assert np.isclose(nilai['z2'], 2.5), 'logit belum benar'
assert np.isclose(nilai['loss'], 0.0788897, atol=1e-6), 'loss belum benar'
print('forward pass sesuai Kasus 1')

{'z1': array([1.5, 1. ]), 'h': array([1.5, 1. ]), 'z2': np.float64(2.5), 'p': np.float64(0.924142), 'loss': np.float64(0.07889)}
forward pass sesuai Kasus 1


In [6]:
def backward(x, W1, W2, y, nilai):
    """TODO 2: kembalikan dict gradien untuk 'W1', 'b1', 'W2', 'b2'.

    Urutan pengerjaan: dz2 -> (dW2, db2, dh) -> dz1 -> (dW1, db1).
    Ingat gradien lokal ReLU dan bentuk perkalian luar untuk dW1.
    """
    h  = nilai['h']
    z1 = nilai['z1']
    p  = nilai['p']

    # 1) Gradien lokal BCE-logits terhadap logit z2: p - y  (skalar)
    dz2 = p - y

    # 2) Gradien terhadap W2: dW2 = dz2 * h   -> shape (2,)
    dW2 = dz2 * h

    # 3) Gradien terhadap b2: db2 = dz2       -> skalar
    db2 = dz2

    # 4) Gradien terhadap h (aktivasi ReLU): dh = dz2 * W2   -> shape (2,)
    dh = dz2 * W2

    # 5) Gradien lokal ReLU: mask = 1[z1 > 0], lalu dz1 = dh * mask
    mask = (z1 > 0).astype(z1.dtype)
    dz1 = dh * mask                          # -> shape (2,)

    # 6) Gradien terhadap W1: perkalian luar x (baris) dengan dz1 (baris)
    #    dW1[i,j] = x[i] * dz1[j]  -> shape (2,2)
    dW1 = np.outer(x, dz1)

    # 7) Gradien terhadap b1: db1 = dz1        -> shape (2,)
    db1 = dz1

    return {'W1': dW1, 'b1': db1, 'W2': dW2, 'b2': db2}

grad_manual = backward(x, W1, W2, y, nilai)
for nama, v in grad_manual.items():
    print(f'{nama:>3}: {np.round(v, 7)}')

# Pemeriksaan wajib: dua angka kunci dari modul.
assert np.allclose(grad_manual['W2'], [-0.1137873, -0.0758582], atol=1e-6)
assert grad_manual['W1'].shape == W1.shape, 'shape dW1 harus sama dengan W1'
print('gradien manual sesuai angka acuan')

 W1: [[-0.3034327  0.1517164]
 [ 0.1517164 -0.0758582]]
 b1: [-0.1517164  0.0758582]
 W2: [-0.1137873 -0.0758582]
 b2: -0.0758582
gradien manual sesuai angka acuan


## C. Autograd - 20 poin

Bagian ini membangun ulang Kasus 1 dengan tensor PyTorch bertipe `float64` dan
`requires_grad=True`, lalu membandingkan gradien hasil autograd dengan gradien
hasil turunan manual pada bagian B.

### Forward pass (NumPy vs PyTorch)

| Besaran | NumPy (bagian B) | PyTorch (autograd) |
|---|---|---|
| $\mathbf{z}^{(1)}$ | `[1.5, 1.0]` | `tensor([1.5000, 1.0000])` |
| $\mathbf{h}$ | `[1.5, 1.0]` | `tensor([1.5000, 1.0000])` |
| $z^{(2)}$ | `2.5` | `tensor(2.5000)` |
| $p$ | `0.924142` | `tensor(0.9241)` |
| $\mathcal{L}$ | `0.07889` | `tensor(0.0789)` |

Nilai forward pass identik, menandakan implementasi NumPy dan PyTorch konsisten.

### Gradien hasil autograd

```python
W1: [[-0.3034327  0.1517164]
     [ 0.1517164 -0.0758582]]
b1: [-0.1517164  0.0758582]
W2: [-0.1137873 -0.0758582]
b2: -0.0758582

In [7]:
tW1 = torch.tensor(W1, requires_grad=True)
tb1 = torch.tensor(b1, requires_grad=True)
tW2 = torch.tensor(W2, requires_grad=True)
tb2 = torch.tensor(b2, requires_grad=True)
tx, ty = torch.tensor(x), torch.tensor(y)
kriteria = nn.BCEWithLogitsLoss()

def forward_torch():
    """TODO 3: hitung loss dengan BCEWithLogitsLoss pada LOGIT."""
    # Linear 1: z1 = x @ W1.T + b1
    z1 = tx @ tW1.T + tb1

    # ReLU: h = max(0, z1)
    h = torch.relu(z1)

    # Linear 2 (tanpa sigmoid!): z2 = h @ W2.T + b2
    # tW2 berbentuk (2,), maka h @ tW2 menghasilkan skalar
    z2 = h @ tW2 + tb2

    # BCEWithLogitsLoss: input = LOGIT (z2), target = y
    loss = kriteria(z2, ty)
    return loss

loss = forward_torch()
loss.backward()

for nama, t in [('W1', tW1), ('b1', tb1), ('W2', tW2), ('b2', tb2)]:
    selisih = np.max(np.abs(t.grad.numpy() - grad_manual[nama]))
    print(f'{nama}: autograd = {np.round(t.grad.numpy(), 7)}   selisih maks = {selisih:.2e}')
    assert selisih < 1e-10, f'gradien {nama} belum cocok dengan hasil manual'
print('autograd cocok dengan backward manual')

W1: autograd = [[-0.3034327  0.1517164]
 [ 0.1517164 -0.0758582]]   selisih maks = 0.00e+00
b1: autograd = [-0.1517164  0.0758582]   selisih maks = 0.00e+00
W2: autograd = [-0.1137873 -0.0758582]   selisih maks = 0.00e+00
b2: autograd = -0.0758582   selisih maks = 0.00e+00
autograd cocok dengan backward manual


In [11]:
# TODO 4: panggil backward() sekali lagi TANPA menghapus gradien,
#         cetak tW2.grad, lalu hapus gradien dan hitung ulang.
#         Jelaskan hasilnya pada sel markdown di bawah.

# Graph dari backward() sebelumnya sudah dibebaskan (retain_graph default=False),
# jadi kita bangun ULANG graph yang identik untuk percobaan akumulasi.
tW1.grad = None; tb1.grad = None; tW2.grad = None; tb2.grad = None

loss2 = forward_torch()
loss2.backward(retain_graph=True)             # backward ke-1, tahan graph
grad_setelah_1x = tW2.grad.detach().clone()
print('setelah backward ke-1 (TANPA zero_grad):')
print(f'  tW2.grad = {np.round(tW2.grad.numpy(), 7)}')

loss2.backward()                               # backward ke-2, akumulasi
print('\nsetelah backward ke-2 (TANPA zero_grad):')
print(f'  tW2.grad = {np.round(tW2.grad.numpy(), 7)}')

dua_kali = grad_setelah_1x * 2
assert np.allclose(tW2.grad.numpy(), dua_kali.numpy(), atol=1e-12), \
    'gradien seharusnya terakumulasi (2x lipat)'
print('\nterbukti: gradien TERAKUMULASI (2x lipat), bukan ditimpa')

# --- Reset + forward/backward ulang untuk verifikasi kebersihan ---
tW1.grad = None; tb1.grad = None; tW2.grad = None; tb2.grad = None

loss3 = forward_torch()
loss3.backward()

print('\nsetelah zero_grad + forward+backward ulang:')
for nama, t in [('W1', tW1), ('b1', tb1), ('W2', tW2), ('b2', tb2)]:
    selisih = np.max(np.abs(t.grad.numpy() - grad_manual[nama]))
    print(f'  {nama}: {np.round(t.grad.numpy(), 7)}   selisih = {selisih:.2e}')
    assert selisih < 1e-10, f'gradien {nama} setelah reset tidak cocok'

print('\nsetelah zero_grad, gradien kembali bersih sesuai hasil manual')

setelah backward ke-1 (TANPA zero_grad):
  tW2.grad = [-0.1137873 -0.0758582]

setelah backward ke-2 (TANPA zero_grad):
  tW2.grad = [-0.2275745 -0.1517164]

terbukti: gradien TERAKUMULASI (2x lipat), bukan ditimpa

setelah zero_grad + forward+backward ulang:
  W1: [[-0.3034327  0.1517164]
 [ 0.1517164 -0.0758582]]   selisih = 0.00e+00
  b1: [-0.1517164  0.0758582]   selisih = 0.00e+00
  W2: [-0.1137873 -0.0758582]   selisih = 0.00e+00
  b2: -0.0758582   selisih = 0.00e+00

setelah zero_grad, gradien kembali bersih sesuai hasil manual


**Penjelasan akumulasi gradien:**

Setelah `backward()` kedua dipanggil tanpa `zero_grad()`, nilai `tW2.grad` menjadi **2× lipat** dari nilai semula, yaitu:

$$
[-0.1137873,\ -0.0758582] \;\longrightarrow\; [-0.2275745,\ -0.1517164]
$$

Hal ini terjadi karena PyTorch **menjumlahkan** gradien baru ke nilai `.grad` yang sudah ada, bukan menimpanya:

$$
\texttt{.grad} \leftarrow \texttt{.grad} + \nabla\mathcal{L}
$$

Karena `loss2` dan graph-nya identik pada kedua pemanggilan, hasilnya tepat $2\nabla$. Setelah gradien direset dengan `.grad = None` (setara dengan `optimizer.zero_grad()`) lalu `backward()` dipanggil ulang, nilai gradien kembali bersih dan cocok persis dengan hasil turunan manual di bagian B (`selisih = 0.00e+00` untuk semua parameter).

**Hubungan dengan `optimizer.zero_grad()` pada training loop:**

Dalam *training loop*, `optimizer.zero_grad()` berperan **wajib** untuk menghapus akumulasi gradien dari iterasi sebelumnya sebelum `loss.backward()` dipanggil. Urutan standarnya:

```python
for xb, yb in loader:
    optimizer.zero_grad()   # 1. bersihkan gradien lama
    loss = model(xb, yb)    # 2. forward pass
    loss.backward()         # 3. hitung gradien baru
    optimizer.step()        # 4. update parameter

## D. Gradient checking - 20 poin

Bandingkan gradien analitik dengan selisih terpusat:

$$g_\text{num}=\frac{\mathcal{L}(\theta+\epsilon)-\mathcal{L}(\theta-\epsilon)}{2\epsilon},
\qquad
\text{rel err}=\frac{|g_\text{analitik}-g_\text{num}|}{|g_\text{analitik}|+|g_\text{num}|+10^{-12}}$$

Ambang lulus: seluruh baris di bawah $10^{-5}$.

In [12]:
EPS = 1e-5

# Parameter Kasus 1 (acuan), dipakai sebagai basis untuk menggeser satu komponen
_PARAMS = {'W1': W1, 'b1': b1, 'W2': W2, 'b2': b2}

def loss_dengan(param, i, delta):
    """TODO 5: salin parameter, geser satu komponen sebesar delta, kembalikan loss."""
    # Salin seluruh parameter agar tidak mengubah nilai asli
    W1c = W1.copy()
    b1c = b1.copy()
    W2c = W2.copy()
    b2c = float(b2)

    # Geser hanya satu komponen pada parameter yang diminta
    if param == 'W1':
        W1c[i] += delta
    elif param == 'b1':
        b1c[i] += delta
    elif param == 'W2':
        W2c[i] += delta
    elif param == 'b2':
        b2c += delta
    else:
        raise ValueError(f'parameter tidak dikenal: {param}')

    # Hitung loss dengan parameter yang sudah digeser
    return forward(x, W1c, b1c, W2c, b2c, y)['loss']

def finite_difference(param, i):
    """TODO 6: kembalikan gradien numerik dengan selisih terpusat."""
    lp = loss_dengan(param, i,  EPS)   # L(theta + eps)
    lm = loss_dengan(param, i, -EPS)   # L(theta - eps)
    return (lp - lm) / (2.0 * EPS)

indeks = ([('W1', (0, 0)), ('W1', (0, 1)), ('W1', (1, 0)), ('W1', (1, 1))]
          + [('b1', (0,)), ('b1', (1,))]
          + [('W2', (0,)), ('W2', (1,))]
          + [('b2', ())])

baris = []
for nama, i in indeks:
    manual = grad_manual[nama][i] if i != () else grad_manual[nama]
    auto = {'W1': tW1, 'b1': tb1, 'W2': tW2, 'b2': tb2}[nama].grad.numpy()
    auto = auto[i] if i != () else auto
    numerik = finite_difference(nama, i)
    rel = abs(manual - numerik) / (abs(manual) + abs(numerik) + 1e-12)
    baris.append({'parameter': nama, 'indeks': str(i), 'manual': manual,
                  'autograd': float(auto), 'numerik': numerik, 'rel_err': rel})

tabel = pd.DataFrame(baris)
print(tabel.to_string(index=False))
print('\nrelative error maksimum:', tabel['rel_err'].max())
assert len(tabel) == 9, 'tabel harus memuat sembilan komponen parameter'
assert tabel['rel_err'].max() < 1e-5, 'masih ada baris yang melampaui ambang'

parameter indeks      manual    autograd     numerik        rel_err
       W1 (0, 0) -0.30343272 -0.30343272 -0.30343272 1.00453007e-10
       W1 (0, 1)  0.15171636  0.15171636  0.15171636 2.95622632e-11
       W1 (1, 0)  0.15171636  0.15171636  0.15171636 2.95622632e-11
       W1 (1, 1) -0.07585818 -0.07585818 -0.07585818 5.73360674e-11
       b1   (0,) -0.15171636 -0.15171636 -0.15171636 2.95622632e-11
       b1   (1,)  0.07585818  0.07585818  0.07585818 5.73360674e-11
       W2   (0,) -0.11378727 -0.11378727 -0.11378727 6.76755052e-11
       W2   (1,) -0.07585818 -0.07585818 -0.07585818 5.73360674e-11
       b2     () -0.07585818 -0.07585818 -0.07585818 5.73360674e-11

relative error maksimum: 1.004530066507852e-10


In [19]:
# TODO 7: simpan tabel ke M02_NIM_metrics.csv
tabel_out = tabel.assign(run_id='gradcheck', seed=SEED)
# Atur urutan kolom: run_id, seed di depan
tabel_out = tabel_out[['run_id', 'seed'] + [c for c in tabel_out.columns
                                            if c not in ('run_id', 'seed')]]

path = f'M02_{NIM}_metrics.csv'
tabel_out.to_csv(path, index=False)
print(f'tersimpan ke {path}')
print('shape:', tabel_out.shape)

tersimpan ke M02_122450058_metrics.csv
shape: (9, 8)


**Checkpoint menit ke-95.** Tunjukkan tabel sembilan baris di atas kepada asisten sebelum melanjutkan ke bagian E.

## E. Diagnosis training loop - 20 poin

Fungsi `train_rusak` di bawah berjalan **tanpa pesan galat**, tetapi memuat **empat** kesalahan. Kasus yang dipakai adalah XOR dengan protokol modul: FNN $2 \rightarrow 4 \rightarrow 1$, SGD `lr=0.1`, 400 epoch, satu batch penuh.

In [20]:
X_xor = torch.tensor([[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]])
y_xor = torch.tensor([[0.0], [1.0], [1.0], [0.0]])

def train_rusak(epoch: int = 400, lr: float = 0.1):
    seed_everything(SEED)
    model = nn.Sequential(
        nn.Linear(2, 4),
        nn.Linear(4, 1),
    )
    opt = torch.optim.SGD(model.parameters(), lr=lr)
    kriteria = nn.BCEWithLogitsLoss()
    riwayat = []

    for _ in range(epoch):
        logits = model(X_xor)
        loss = kriteria(torch.sigmoid(logits), y_xor)
        opt.step()
        loss.backward()
        riwayat.append(loss.item())
    return model, riwayat

model_rusak, riwayat_rusak = train_rusak()
print(f'loss awal  : {riwayat_rusak[0]:.4f}')
print(f'loss akhir : {riwayat_rusak[-1]:.4f}')
with torch.no_grad():
    print('prediksi   :', (torch.sigmoid(model_rusak(X_xor)) > 0.5).int().flatten().tolist())
    print('target     :', y_xor.int().flatten().tolist())

RuntimeError: one of the variables needed for gradient computation has been modified by an inplace operation: [torch.DoubleTensor [4, 1]], which is output 0 of AsStrided, is at version 2; expected version 1 instead. Hint: enable anomaly detection to find the operation that failed to compute its gradient, with torch.autograd.set_detect_anomaly(True, check_nan=False).

### Temuan kesalahan

| No | Baris kode bermasalah | Mengapa keliru | Gejala yang terlihat |
|----|----------------------|----------------|----------------------|
| 1 | Tidak ada `opt.zero_grad()` di loop | Gradien terakumulasi antar iterasi, tidak pernah di-reset | Loss tidak turun stabil, bisa `nan` |
| 2 | `opt.step()` sebelum `loss.backward()` | Parameter diubah in-place sebelum backward; graph autograd pakai versi lama | `RuntimeError: inplace operation ... expected version 1 instead` |
| 3 | `kriteria(torch.sigmoid(logits), y_xor)` | `BCEWithLogitsLoss` sudah ada sigmoid internal → sigmoid ganda, gradien salah | Loss stagnan di ~0.69, prediksi tak bisa pisahkan kelas |
| 4 | `nn.Sequential(nn.Linear(2,4), nn.Linear(4,1))` tanpa aktivasi | Dua Linear ekuivalen satu Linear → tidak bisa belajar XOR | Akurasi ~50%, prediksi konstan |

In [22]:
# TODO 8: perbaiki SATU PER SATU. Salin train_rusak, perbaiki satu kesalahan,
#         jalankan, lalu catat loss akhirnya. Ulangi sampai keempatnya beres.
#         Simpan setiap tahap ke daftar berikut.

tahap = []

def train_versi(epoch=400, lr=0.1,
                fix_zero_grad=False,
                fix_urutan=False,
                fix_sigmoid=False,
                fix_aktivasi=False):
    """Salinan train_rusak dengan flag perbaikan satu per satu."""
    seed_everything(SEED)

    # Susun arsitektur: dengan / tanpa aktivasi
    if fix_aktivasi:
        model = nn.Sequential(nn.Linear(2, 4), nn.ReLU(), nn.Linear(4, 1))
    else:
        model = nn.Sequential(nn.Linear(2, 4), nn.Linear(4, 1))

    opt = torch.optim.SGD(model.parameters(), lr=lr)
    kriteria = nn.BCEWithLogitsLoss()
    riwayat = []

    for _ in range(epoch):
        if fix_zero_grad:
            opt.zero_grad()

        logits = model(X_xor)

        if fix_sigmoid:
            loss = kriteria(logits, y_xor)          # logit mentah
        else:
            loss = kriteria(torch.sigmoid(logits), y_xor)  # sigmoid ganda

        if fix_urutan:
            loss.backward()
            opt.step()
        else:
            opt.step()          # urutan salah (seperti train_rusak)
            loss.backward()

        riwayat.append(loss.item())

    # Prediksi akhir
    with torch.no_grad():
        pred = (torch.sigmoid(model(X_xor)) > 0.5).int().flatten().tolist()
    target = y_xor.int().flatten().tolist()
    benar = sum(p == t for p, t in zip(pred, target))

    return riwayat, pred, benar


# Jalankan lima konfigurasi: 0 perbaikan, lalu tambah 1 perbaikan tiap tahap
konfigurasi = [
    ('perbaikan-0 (baseline)', dict()),
    ('perbaikan-1: zero_grad', dict(fix_zero_grad=True)),
    ('perbaikan-2: + urutan backward/step', dict(fix_zero_grad=True, fix_urutan=True)),
    ('perbaikan-3: + logit mentah', dict(fix_zero_grad=True, fix_urutan=True, fix_sigmoid=True)),
    ('perbaikan-4: + aktivasi ReLU', dict(fix_zero_grad=True, fix_urutan=True,
                                          fix_sigmoid=True, fix_aktivasi=True)),
]

for nama, kwargs in konfigurasi:
    try:
        riwayat, pred, benar = train_versi(**kwargs)
        tahap.append({
            'tahap': nama,
            'loss_awal': round(riwayat[0], 6),
            'loss_akhir': round(riwayat[-1], 6),
            'prediksi': pred,
            'benar': benar,
        })
        print(f'{nama:45s} loss: {riwayat[0]:.4f} -> {riwayat[-1]:.4f}  '
              f'benar: {benar}/4  pred: {pred}')
    except RuntimeError as e:
        tahap.append({
            'tahap': nama,
            'loss_awal': None,
            'loss_akhir': None,
            'prediksi': None,
            'benar': 0,
            'error': str(e)[:60],
        })
        print(f'{nama:45s} ERROR: {str(e)[:60]}...')

pd.DataFrame(tahap)

perbaikan-0 (baseline)                        ERROR: one of the variables needed for gradient computation has bee...
perbaikan-1: zero_grad                        loss: 0.7075 -> 0.7075  benar: 2/4  pred: [0, 0, 0, 0]
perbaikan-2: + urutan backward/step           loss: 0.7075 -> 0.6932  benar: 2/4  pred: [0, 0, 0, 0]
perbaikan-3: + logit mentah                   loss: 0.7587 -> 0.6933  benar: 2/4  pred: [1, 1, 0, 0]
perbaikan-4: + aktivasi ReLU                  loss: 0.7509 -> 0.1869  benar: 4/4  pred: [0, 1, 1, 0]


,tahap,loss_awal,loss_akhir,prediksi,benar,error
0,perbaikan-0 (baseline),NaN,NaN,None,0,one of the variables needed for gradient compu...
1,perbaikan-1: zero_grad,0.707461,0.707461,"[0, 0, 0, 0]",2,NaN
2,perbaikan-2: + urutan backward/step,0.707461,0.693224,"[0, 0, 0, 0]",2,NaN
3,perbaikan-3: + logit mentah,0.758698,0.693295,"[1, 1, 0, 0]",2,NaN
4,perbaikan-4: + aktivasi ReLU,0.750930,0.186892,"[0, 1, 1, 0]",4,NaN


In [26]:
def train_benar(epoch: int = 2000, lr: float = 0.5):
    """Versi bebas dari keempat kesalahan."""
    seed_everything(SEED)
    torch.set_default_dtype(torch.float64)

    model = nn.Sequential(
        nn.Linear(2, 4),
        nn.ReLU(),
        nn.Linear(4, 1),
    ).double()

    opt = torch.optim.SGD(model.parameters(), lr=lr)
    kriteria = nn.BCEWithLogitsLoss()
    riwayat = []

    for _ in range(epoch):
        opt.zero_grad()
        logits = model(X_xor.double())
        loss = kriteria(logits, y_xor.double())
        loss.backward()
        opt.step()
        riwayat.append(loss.item())

    return model, riwayat


model_benar, riwayat_benar = train_benar()
print(f'loss akhir: {riwayat_benar[-1]:.4f}')
with torch.no_grad():
    prediksi = (torch.sigmoid(model_benar(X_xor.double())) > 0.5).int().flatten()
print('prediksi  :', prediksi.tolist())

assert riwayat_benar[-1] < 0.1, 'loss akhir harus di bawah 0,1'
assert torch.equal(prediksi, y_xor.int().flatten()), 'keempat titik XOR harus benar'
print('training loop sudah benar')

loss akhir: 0.0007
prediksi  : [0, 1, 1, 0]
training loop sudah benar


In [27]:
# TODO 10: gabungkan catatan tahap perbaikan ke metrics.csv.
df_tahap = pd.DataFrame(tahap)
df_tahap.insert(0, 'seed', SEED)
df_tahap.to_csv(f'M02_{NIM}_metrics_loop.csv', index=False)
print(df_tahap.to_string(index=False))

 seed                               tahap  loss_awal  loss_akhir     prediksi  benar                                                        error
   58              perbaikan-0 (baseline)        NaN         NaN         None      0 one of the variables needed for gradient computation has bee
   58              perbaikan-1: zero_grad   0.707461    0.707461 [0, 0, 0, 0]      2                                                          NaN
   58 perbaikan-2: + urutan backward/step   0.707461    0.693224 [0, 0, 0, 0]      2                                                          NaN
   58         perbaikan-3: + logit mentah   0.758698    0.693295 [1, 1, 0, 0]      2                                                          NaN
   58        perbaikan-4: + aktivasi ReLU   0.750930    0.186892 [0, 1, 1, 0]      4                                                          NaN


## F. Tugas individu

### 1. Perluasan jaringan (3 neuron hidden)

**Arsitektur baru:** $2 \to 3 \to 1$.

$$
\mathbf{W}^{(1)}=\begin{bmatrix}0.5 & -0.5\\ 1 & 1\\ -1 & 0.5\end{bmatrix},\quad
\mathbf{b}^{(1)}=\begin{bmatrix}0 & 0 & 0.25\end{bmatrix},\quad
\mathbf{W}^{(2)}=\begin{bmatrix}2 & -1 & -0.5\end{bmatrix},\quad
b^{(2)}=0.5
$$

**Turunan manual (ringkas):**

1. $z^{(2)} = \mathbf{h}\mathbf{W}^{(2)\top} + b^{(2)}$, $\partial\mathcal{L}/\partial z^{(2)} = p - y$
2. $\partial\mathcal{L}/\partial \mathbf{W}^{(2)} = (p-y)\,\mathbf{h}$ — shape `(3,)`
3. $\partial\mathcal{L}/\partial b^{(2)} = p - y$ — skalar
4. $\partial\mathcal{L}/\partial \mathbf{h} = (p-y)\,\mathbf{W}^{(2)}$ — shape `(3,)`
5. $\partial\mathcal{L}/\partial \mathbf{z}^{(1)} = \partial\mathcal{L}/\partial \mathbf{h} \odot \mathbb{1}[\mathbf{z}^{(1)}>0]$ — shape `(3,)`
6. $\partial\mathcal{L}/\partial \mathbf{W}^{(1)} = \mathbf{x}^\top (\partial\mathcal{L}/\partial \mathbf{z}^{(1)})$ — shape `(3,2)`
7. $\partial\mathcal{L}/\partial \mathbf{b}^{(1)} = \partial\mathcal{L}/\partial \mathbf{z}^{(1)}$ — shape `(3,)`


In [31]:
# --- Kasus 1 diperluas: 2 -> 3 -> 1 ---
W1e = np.array([[0.5, -0.5],
                [1.0,  1.0],
                [-1.0, 0.5]])
b1e = np.array([0.0, 0.0, 0.25])
W2e = np.array([2.0, -1.0, -0.5])
b2e = 0.5

def forward_ext(x, W1, b1, W2, b2, y):
    z1 = x @ W1.T + b1
    h  = np.maximum(0.0, z1)
    z2 = h @ W2 + b2
    p  = 1/(1+np.exp(-z2))
    loss = -(y*np.log(p) + (1-y)*np.log(1-p))
    return {'z1': z1, 'h': h, 'z2': z2, 'p': p, 'loss': loss}

def backward_ext(x, W1, W2, y, nilai):
    p, h, z1 = nilai['p'], nilai['h'], nilai['z1']
    dz2 = p - y
    dW2 = dz2 * h
    db2 = dz2
    dh  = dz2 * W2
    dz1 = dh * (z1 > 0).astype(z1.dtype)
    dW1 = np.outer(dz1, x)      # shape (3,2): dW1[k,j] = dz1[k]*x[j]
    db1 = dz1
    return {'W1': dW1, 'b1': db1, 'W2': dW2, 'b2': db2}

nilai_e = forward_ext(x, W1e, b1e, W2e, b2e, y)
grad_e  = backward_ext(x, W1e, W2e, y, nilai_e)

# --- Gradient checking central difference ---
EPS = 1e-5

def loss_e(W1, b1, W2, b2):
    return forward_ext(x, W1, b1, W2, b2, y)['loss']

def num_grad(param, i):
    if param == 'W1':
        Wp = W1e.copy(); Wm = W1e.copy()
        Wp[i] += EPS; Wm[i] -= EPS
        return (loss_e(Wp, b1e, W2e, b2e) - loss_e(Wm, b1e, W2e, b2e)) / (2*EPS)
    if param == 'b1':
        bp = b1e.copy(); bm = b1e.copy()
        bp[i] += EPS; bm[i] -= EPS
        return (loss_e(W1e, bp, W2e, b2e) - loss_e(W1e, bm, W2e, b2e)) / (2*EPS)
    if param == 'W2':
        Wp = W2e.copy(); Wm = W2e.copy()
        Wp[i] += EPS; Wm[i] -= EPS
        return (loss_e(W1e, b1e, Wp, b2e) - loss_e(W1e, b1e, Wm, b2e)) / (2*EPS)
    if param == 'b2':
        return (loss_e(W1e, b1e, W2e, b2e + EPS)
                - loss_e(W1e, b1e, W2e, b2e - EPS)) / (2*EPS)

# --- Tabel rel err ---
idx = ([('W1', (i, j)) for i in range(3) for j in range(2)]
       + [('b1', (i,)) for i in range(3)]
       + [('W2', (i,)) for i in range(3)]
       + [('b2', ())])

rows = []
for nama, i in idx:
    a = grad_e[nama][i] if i != () else grad_e[nama]
    n = num_grad(nama, i)
    rows.append({
        'param': nama,
        'indeks': str(i),
        'analitik': a,
        'numerik': n,
        'rel_err': abs(a - n) / (abs(a) + abs(n) + 1e-12),
    })

tabel_e = pd.DataFrame(rows)
print(tabel_e.to_string(index=False))
print(f'\nrel err maksimum: {tabel_e["rel_err"].max():.3e}')

assert len(tabel_e) == 13, 'tabel harus memuat 13 komponen parameter'
assert tabel_e['rel_err'].max() < 1e-5, 'ada baris yang melampaui ambang 1e-5'
print('semua baris lulus ambang 1e-5')

tabel_e.to_csv(f'M02_{122450058}_metrics_ext.csv', index=False)
print(f'tersimpan ke M02_{122450058}_metrics_ext.csv')

param indeks    analitik     numerik        rel_err
   W1 (0, 0) -0.30343272 -0.30343272 1.00453007e-10
   W1 (0, 1)  0.15171636  0.15171636 2.95622632e-11
   W1 (1, 0)  0.15171636  0.15171636 2.95622632e-11
   W1 (1, 1) -0.07585818 -0.07585818 5.73360674e-11
   W1 (2, 0)  0.00000000  0.00000000 0.00000000e+00
   W1 (2, 1) -0.00000000  0.00000000 0.00000000e+00
   b1   (0,) -0.15171636 -0.15171636 2.95622632e-11
   b1   (1,)  0.07585818  0.07585818 5.73360674e-11
   b1   (2,)  0.00000000  0.00000000 0.00000000e+00
   W2   (0,) -0.11378727 -0.11378727 6.76755052e-11
   W2   (1,) -0.07585818 -0.07585818 5.73360674e-11
   W2   (2,) -0.00000000  0.00000000 0.00000000e+00
   b2     () -0.07585818 -0.07585818 5.73360674e-11

rel err maksimum: 1.005e-10
semua baris lulus ambang 1e-5
tersimpan ke M02_122450058_metrics_ext.csv


## G. Pertanyaan analisis

1. Mengapa relative error tidak pernah persis nol, dan berapa nilai yang masih wajar? TODO
2. Apa yang terjadi pada tabel bila $\epsilon = 10^{-9}$? Jalankan dan jelaskan. TODO
3. Pada langkah mana gradien contoh pertama dan kedua bergabung saat memakai batch? TODO
4. Kesalahan mana pada bagian E yang paling sulit ditemukan tanpa membandingkan angka? TODO
5. Apa beda peran backpropagation dan optimizer? (maksimal tiga kalimat) TODO

### 1. Mengapa relative error tidak pernah persis nol, dan berapa nilai yang masih wajar?

Relative error tidak pernah persis nol karena perhitungan dilakukan dalam **aritmetika floating-point terbatas** (`float64` ≈ 15–16 digit signifikan). Dua sumber galat selalu ada:

- **Truncation error** dari selisih terpusat: aproksimasi $\frac{L(\theta+\epsilon)-L(\theta-\epsilon)}{2\epsilon}$ memiliki galat orde $O(\epsilon^2)$ karena mengabaikan suku turunan ketiga.
- **Round-off error** dari representasi bilangan: pembilang adalah selisih dua nilai loss yang sangat dekat (beda $\sim\epsilon$), sehingga digit signifikan yang tersisa terbatas; galat ini tumbuh seperti $O(\text{machine-eps}/\epsilon)$.

Nilai yang masih **wajar** untuk `float64` dengan $\epsilon = 10^{-5}$–$10^{-6}$ adalah rel err di orde $10^{-8}$ hingga $10^{-12}$. Ambang lulus modul $10^{-5}$ sudah cukup longgar; rel err di atas $10^{-4}$ biasanya menandakan bug pada gradien analitik.

### 2. Apa yang terjadi pada tabel bila $\epsilon = 10^{-9}$?

**Kode eksperimen lengkap** (dijalankan di sel baru setelah F.1):

In [32]:
def gradient_check_ext(eps):
    """Jalankan gradient checking untuk arsitektur 2->3->1 dengan epsilon tertentu."""
    def loss_e(W1, b1, W2, b2):
        return forward_ext(x, W1, b1, W2, b2, y)['loss']

    def num_grad(param, i):
        if param == 'W1':
            Wp = W1e.copy(); Wm = W1e.copy()
            Wp[i] += eps; Wm[i] -= eps
            return (loss_e(Wp, b1e, W2e, b2e) - loss_e(Wm, b1e, W2e, b2e)) / (2*eps)
        if param == 'b1':
            bp = b1e.copy(); bm = b1e.copy()
            bp[i] += eps; bm[i] -= eps
            return (loss_e(W1e, bp, W2e, b2e) - loss_e(W1e, bm, W2e, b2e)) / (2*eps)
        if param == 'W2':
            Wp = W2e.copy(); Wm = W2e.copy()
            Wp[i] += eps; Wm[i] -= eps
            return (loss_e(W1e, b1e, Wp, b2e) - loss_e(W1e, b1e, Wm, b2e)) / (2*eps)
        if param == 'b2':
            return (loss_e(W1e, b1e, W2e, b2e + eps)
                    - loss_e(W1e, b1e, W2e, b2e - eps)) / (2*eps)

    idx = ([('W1', (i, j)) for i in range(3) for j in range(2)]
           + [('b1', (i,)) for i in range(3)]
           + [('W2', (i,)) for i in range(3)]
           + [('b2', ())])

    rows = []
    for nama, i in idx:
        a = grad_e[nama][i] if i != () else grad_e[nama]
        n = num_grad(nama, i)
        rows.append({
            'param': nama,
            'indeks': str(i),
            'analitik': a,
            'numerik': n,
            'rel_err': abs(a - n) / (abs(a) + abs(n) + 1e-12),
        })
    return pd.DataFrame(rows)


# --- Jalankan dengan EPS = 1e-5 (baseline) ---
tabel_e_1e5 = gradient_check_ext(1e-5)
print('=== EPS = 1e-5 ===')
print(tabel_e_1e5.to_string(index=False))
print('rel err maks:', tabel_e_1e5['rel_err'].max())

# --- Jalankan dengan EPS = 1e-9 ---
tabel_e_1e9 = gradient_check_ext(1e-9)
print('\n=== EPS = 1e-9 ===')
print(tabel_e_1e9.to_string(index=False))
print('rel err maks:', tabel_e_1e9['rel_err'].max())

# --- Bandingkan ---
print('\n=== Perbandingan rel err maks ===')
print(f'EPS = 1e-5  → rel err maks = {tabel_e_1e5["rel_err"].max():.3e}')
print(f'EPS = 1e-9  → rel err maks = {tabel_e_1e9["rel_err"].max():.3e}')


=== EPS = 1e-5 ===
param indeks    analitik     numerik        rel_err
   W1 (0, 0) -0.30343272 -0.30343272 1.00453007e-10
   W1 (0, 1)  0.15171636  0.15171636 2.95622632e-11
   W1 (1, 0)  0.15171636  0.15171636 2.95622632e-11
   W1 (1, 1) -0.07585818 -0.07585818 5.73360674e-11
   W1 (2, 0)  0.00000000  0.00000000 0.00000000e+00
   W1 (2, 1) -0.00000000  0.00000000 0.00000000e+00
   b1   (0,) -0.15171636 -0.15171636 2.95622632e-11
   b1   (1,)  0.07585818  0.07585818 5.73360674e-11
   b1   (2,)  0.00000000  0.00000000 0.00000000e+00
   W2   (0,) -0.11378727 -0.11378727 6.76755052e-11
   W2   (1,) -0.07585818 -0.07585818 5.73360674e-11
   W2   (2,) -0.00000000  0.00000000 0.00000000e+00
   b2     () -0.07585818 -0.07585818 5.73360674e-11
rel err maks: 1.004530066507852e-10

=== EPS = 1e-9 ===
param indeks    analitik     numerik    rel_err
   W1 (0, 0) -0.30343272 -0.30343271 0.00000002
   W1 (0, 1)  0.15171636  0.15171639 0.00000010
   W1 (1, 0)  0.15171636  0.15171639 0.00000010
   W1

### 3. Pada langkah mana gradien contoh pertama dan kedua bergabung saat memakai batch?

**Data batch:**

$$
\mathbf{X} = \begin{bmatrix}2 & -1\\ -1 & 3\end{bmatrix},\qquad
\mathbf{y} = \begin{bmatrix}1\\ 0\end{bmatrix}
$$

**Loss rata-rata:**

$$
\mathcal{L} = \frac{1}{N}\sum_{i=1}^{N}\mathcal{L}_i = \frac{1}{2}\big(\mathcal{L}_1 + \mathcal{L}_2\big)
$$

**Hasil forward batch:**

$$
\texttt{loss batch} = 0.06688595672531382
$$

**Hasil backward batch:**

$$
\frac{\partial\mathcal{L}}{\partial\mathbf{W}^{(1)}} =
\begin{bmatrix}
-0.1517164 & 0.0758582\\
 0.1025598 & -0.1180341\\
 0.0133508 & -0.0400525
\end{bmatrix},\quad
\frac{\partial\mathcal{L}}{\partial\mathbf{b}^{(1)}} =
\begin{bmatrix}
-0.0758582 & 0.0112274 & -0.0133508
\end{bmatrix}
$$

$$
\frac{\partial\mathcal{L}}{\partial\mathbf{W}^{(2)}} =
\begin{bmatrix}
-0.0568936 & 0.0154742 & 0.0734296
\end{bmatrix},\quad
\frac{\partial\mathcal{L}}{\partial b^{(2)}} = -0.0112274
$$

---

In [34]:
def backward_batch(X, W1, W2, y, nilai):
    H, Z1, P = nilai['H'], nilai['Z1'], nilai['P']
    N = X.shape[0]

    dZ2 = (P - y) / N
    dW2 = dZ2 @ H
    db2 = dZ2.sum()
    dH  = np.outer(dZ2, W2)
    dZ1 = dH * (Z1 > 0)
    dW1 = dZ1.T @ X
    db1 = dZ1.sum(axis=0)
    return {'W1': dW1, 'b1': db1, 'W2': dW2, 'b2': db2}


# === PANGGIL FUNGSINYA ===
Xb = np.array([[2.0, -1.0],
               [-1.0, 3.0]])
yb = np.array([1.0, 0.0])

# Hitung forward dulu untuk mendapatkan nilai H, Z1, P
def forward_batch(X, W1, b1, W2, b2, y):
    Z1 = X @ W1.T + b1
    H  = np.maximum(0, Z1)
    Z2 = H @ W2 + b2
    P  = 1/(1+np.exp(-Z2))
    loss = -np.mean(y*np.log(P) + (1-y)*np.log(1-P))
    return {'Z1': Z1, 'H': H, 'Z2': Z2, 'P': P, 'loss': loss}

nilai_b = forward_batch(Xb, W1e, b1e, W2e, b2e, yb)
grad_b  = backward_batch(Xb, W1e, W2e, yb, nilai_b)

# === CETAK HASIL ===
print('loss batch:', nilai_b['loss'])
print('\nGradien batch:')
for k, v in grad_b.items():
    print(f'  {k}: {np.round(v, 7)}')

loss batch: 0.06688595672531382

Gradien batch:
  W1: [[-0.1517164  0.0758582]
 [ 0.1025598 -0.1180341]
 [ 0.0133508 -0.0400525]]
  b1: [-0.0758582  0.0112274 -0.0133508]
  W2: [-0.0568936  0.0154742  0.0734296]
  b2: -0.0112274


### 4. Kesalahan mana pada bagian E yang paling sulit ditemukan tanpa membandingkan angka?

**Jawaban: Bug #4 — tidak ada aktivasi non-linear.**

**Perbandingan tingkat kesulitan deteksi:**

| Bug | Gejala permukaan | Terdeteksi tanpa angka? |
|---|---|---|
| #1 Tidak ada `zero_grad()` | Loss tidak stabil, bisa `nan` | **Mudah** — loss jelas kacau |
| #2 `step()` sebelum `backward()` | `RuntimeError: inplace operation` | **Sangat mudah** — langsung error |
| #3 Sigmoid ganda | Loss stagnan di ~0.69 | **Sedang** — perlu lihat angka loss |
| **#4 Tanpa aktivasi non-linear** | **Tidak error, loss turun tipis, prediksi sebagian benar** | **Paling sulit** |

**Mengapa bug #4 paling tersembunyi:**

1. **Tidak ada pesan error.** Kode jalan normal, `backward()` sukses, `step()` berjalan, epoch selesai.
2. **Loss tetap turun tipis** — terlihat seperti training "berhasil separuh".
3. **Prediksi sebagian benar** (2 dari 4 pola XOR = setara menebak acak untuk klasifikasi biner seimbang).
4. **Loss berada di *chance level* 0.693** = $-\ln 0.5$. Nilai ini "wajar" untuk model yang tidak belajar, sehingga sulit disadari sebagai bug tanpa tahu bahwa 0.693 adalah *baseline* tebakan.

**Bukti angka yang mengungkap bug #4:**

| Tahap | Perbaikan | loss akhir | benar | Interpretasi |
|---|---|---|---|---|
| 0 | baseline | ERROR | 0/4 | Bug #2 fatal |
| 1 | + `zero_grad()` | 0.7075 | 2/4 | Stagnan |
| 2 | + urutan `backward→step` | **0.6932** | 2/4 | **Chance level** (~0.693) |
| 3 | + logit mentah | **0.6933** | 2/4 | **Masih chance level** |
| 4 | + ReLU | **0.1869** | **4/4** | Berhasil |

Perhatikan tahap 2 dan 3: meskipun bug #1 dan #3 sudah diperbaiki, loss **tetap 0.693** — tidak bergerak. Inilah petunjuk ada bug **struktural** (arsitektur) yang tidak bisa diperbaiki dengan memperbaiki urutan operasi atau formula loss.

**Penjelasan matematis:** dua `nn.Linear` tanpa aktivasi setara dengan satu transformasi linear:

$$
\mathbf{W}^{(2)}(\mathbf{W}^{(1)}\mathbf{x}) = \tilde{\mathbf{W}}\mathbf{x}
$$

Model hanya bisa memisahkan secara **linear**, sedangkan XOR **tidak *linearly separable***. Akibatnya loss berhenti di *chance level*.

**Kesimpulan:** bug #4 paling sulit dideteksi karena tidak error, loss tetap turun tipis, dan prediksi sebagian benar. Satu-satunya cara menemukannya adalah **membandingkan angka loss dengan *chance level* 0.693** — menyadari bahwa loss **tidak pernah turun di bawah 0.69** meskipun gradient sudah benar. Ini menandakan masalah **kapasitas model**, bukan bug operasional.

---

### 5. Apa beda peran backpropagation dan optimizer? (maksimal tiga kalimat)

**Backpropagation** menghitung gradien loss terhadap setiap parameter (mengisi `.grad`) dengan menerapkan *chain rule* secara mundur melalui graph komputasi. **Optimizer** menggunakan gradien tersebut untuk **memperbarui parameter** menurut aturan tertentu, misalnya SGD: $\theta \leftarrow \theta - \eta\nabla_\theta\mathcal{L}$, atau Adam dengan momen adaptif. Jadi backprop = *menghitung arah*, optimizer = *melangkah ke arah itu*; keduanya wajib ada dengan urutan `loss.backward()` → `optimizer.step()`.

- [yes] Identitas, seed, versi library, dan device tercantum.
- [yes] Seluruh `TODO` dan `raise NotImplementedError` sudah diganti.
- [yes] Turunan manual ditulis pada sel markdown bagian B.
- [yes] Tabel relative error memuat sembilan baris dan seluruhnya lulus ambang.
- [yes] Keempat kesalahan bagian E ditemukan, dibuktikan, dan diperbaiki bertahap.
- [yes] Notebook lolos *Restart Kernel and Run All*.
- [yes] Berkas: `M02_122450058.ipynb`, `M02_122450058.pdf`, `M02_122450058_metrics.csv`.